# 06a — Operating policy frozen on internal validation (no test data)

This notebook **never reads the official MURA `valid` split**. It can be re-run after every modelling change;
each run writes a new, time-stamped `operating_policy.json`. The locked test (Notebook 06b) is run **once**,
only after the team freezes the model.

**Candidates:** Model A (Notebook 03) and every Model B run listed in `MODEL_B_RUN_IDS` (Notebook 05 with
different training seeds).

**Pre-specified rules (fixed before looking at the candidate results):**

1. Threshold per model: the highest internal-validation threshold with **sensitivity ≥ 95 %** (team decision of
   15.10.2026); the ≥ 90 % threshold of Notebook 04 is reported for comparison only.
2. Model B for the locked test: the seed run with the highest validation **specificity at sensitivity ≥ 95 %**
   (ties: higher AUROC).
3. Pipeline model: Model B only if its specificity at sensitivity ≥ 95 % exceeds Model A's by at least
   0.02; otherwise the simpler Model A.

Seed-to-seed spread of Model B is reported, so a single lucky run is not mistaken for an improvement.

Outputs: `DATA/RUNS_ARTEM/06A_OPERATING_POLICY/run_…/`. Inputs are read-only.

In [ ]:
import hashlib
import json
import random
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    roc_auc_score,
    roc_curve,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| Torch:", torch.__version__)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

# The team DATA folder is shared from the project owner's Drive. Members see it
# either through a "DATA" shortcut in My Drive or through the shared
# "Final Project DS23" shortcut. Both resolve to the same physical folder.
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DIR_CANDIDATES = [
    DRIVE_ROOT / "DATA",
    DRIVE_ROOT
    / "Final Project DS23"
    / "Agentic AI-Assisted Detection of Musculoskeletal Abnormalities in Emergency X-rays"
    / "DATA",
]


def resolve_data_dir(candidates=DATA_DIR_CANDIDATES):
    for candidate in candidates:
        if (candidate / "MURA-v1.1_files.zip").exists() and (candidate / "WRIST_SPLITS").is_dir():
            return candidate
    raise FileNotFoundError(
        "Team DATA folder not found. Add a 'DATA' shortcut to My Drive or check "
        "that 'Final Project DS23' is in My Drive. Tried:\n"
        + "\n".join(str(c) for c in candidates)
    )


DATA_DIR = resolve_data_dir()
RAW_ZIP = DATA_DIR / "MURA-v1.1_files.zip"
SPLITS_DIR = DATA_DIR / "WRIST_SPLITS"            # read-only here (Notebook 01 output)
PREPROCESSING_DIR = DATA_DIR / "PREPROCESSED_WRIST"  # read-only here (Notebook 02 output)
MODEL_A_DIR = DATA_DIR / "BASELINE_MODEL_A"        # read-only here (Notebook 03 output)

# Shared-folder rule: this notebook never modifies or replaces existing team
# files. Everything it writes goes to a NEW run folder under RUNS_ARTEM.
RUN_ROOT = DATA_DIR / "RUNS_ARTEM"
STAGE_DIR = RUN_ROOT / "06A_OPERATING_POLICY"


def new_run_dir(stage_dir):
    run_id = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
    run_dir = stage_dir / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def out_path(run_dir, name):
    """Return a path inside this run's own folder; refuse anything else."""
    path = run_dir / name
    assert str(path).startswith(str(run_dir) + "/"), path
    assert not path.exists(), f"Refusing to overwrite: {path}"
    return path


print("Team DATA folder:", DATA_DIR)
print("Writes only under:", STAGE_DIR)

## Settings

In [ ]:
# Notebook 05 runs to compare (one per training seed): seed 23 (17.10.2026, executed notebook in notebooks/05_…), seed 42 and seed 7 (16.10.2026).
MODEL_B_RUN_IDS = ["run_20261017T095546Z", "run_20261016T121029Z", "run_20261016T123127Z"]
PRIMARY_SENSITIVITY_TARGET = 0.95    # team decision, 15.10.2026
REFERENCE_SENSITIVITY_TARGET = 0.90  # Notebook 04 policy, reported for comparison only
CLOSE_SPECIFICITY_MARGIN = 0.02      # pre-specified: below this gain the simpler Model A is kept

assert "SET_ME" not in MODEL_B_RUN_IDS, "Fill in the Notebook 05 run ids"
RUN_DIR = new_run_dir(STAGE_DIR)
print("This run writes to:", RUN_DIR)


def sha256_of(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def relative_to_data(path):
    # Stored relative to the team DATA folder, so teammates who reach DATA through a
    # different Drive path (e.g. a "DATA" shortcut) can open the frozen files in 06b and 07.
    return Path(path).relative_to(DATA_DIR).as_posix()

## Load the validation predictions of every candidate

In [ ]:
MODEL_A_FILE = MODEL_A_DIR / "frozen_resnet18_mean_embedding_logreg.joblib"
meta_a = json.loads((MODEL_A_DIR / "run_metadata.json").read_text())
assert meta_a["official_mura_valid_read"] is False

candidates = {"A": {"kind": "A", "predictions_path": MODEL_A_DIR / "validation_predictions.csv"}}
for run_id in MODEL_B_RUN_IDS:
    run_dir = RUN_ROOT / "05_MODEL_B_FINETUNE" / run_id
    assert run_dir.is_dir(), f"Model B run not found: {run_dir}"
    meta = json.loads((run_dir / "run_metadata.json").read_text())
    selection = json.loads((run_dir / "model_selection.json").read_text())
    assert meta["official_mura_valid_read"] is False, run_id
    assert sha256_of(run_dir / "best_model_b.pt") == meta["best_checkpoint_sha256"], f"{run_id}: checkpoint changed"
    name = f"B_seed{meta['config']['seed']}"
    assert name not in candidates, f"Two runs with the same seed: {name}"
    candidates[name] = {
        "kind": "B",
        "run_id": run_id,
        "seed": meta["config"]["seed"],
        "aggregation": selection["selected_aggregation"],
        "best_epoch": selection["best_epoch"],
        "artifact": run_dir / "best_model_b.pt",
        "artifact_sha256": meta["best_checkpoint_sha256"],
        "predictions_path": run_dir / "validation_predictions.csv",
    }

validation = {}
for name, info in candidates.items():
    frame = pd.read_csv(info["predictions_path"])
    assert frame["split"].eq("validation").all(), name
    assert frame["study_id"].is_unique, name
    assert frame["abnormal_probability"].between(0, 1).all(), name
    validation[name] = frame
study_sets = {name: frozenset(frame["study_id"]) for name, frame in validation.items()}
assert len(set(study_sets.values())) == 1, "Candidates were validated on different studies"
print("Candidates:", list(candidates), "| validation studies:", len(validation["A"]))

## Operating points at the pre-specified sensitivity targets

In [ ]:
def classification_metrics(y_true, probabilities, threshold):
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        "threshold": float(threshold),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(precision_score(y_true, predicted, zero_division=0)),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
        "flagged_abnormal": int(predicted.sum()),
        "flagged_percent": float(100 * predicted.mean()),
    }


def highest_threshold_reaching(y_true, probabilities, target):
    # Same rule as Notebook 04: highest validation threshold with sensitivity >= target.
    for threshold in np.unique(probabilities)[::-1]:
        if classification_metrics(y_true, probabilities, threshold)["sensitivity"] >= target:
            return float(threshold)
    raise RuntimeError(f"No threshold reaches sensitivity {target}")



policy_rows = {}
for name, frame in validation.items():
    y = frame["label"].to_numpy(dtype=int)
    p = frame["abnormal_probability"].to_numpy(dtype=float)
    for label, target in (("primary_95", PRIMARY_SENSITIVITY_TARGET), ("reference_90", REFERENCE_SENSITIVITY_TARGET)):
        threshold = highest_threshold_reaching(y, p, target)
        row = classification_metrics(y, p, threshold)
        row.update({"model": name, "policy": label, "target_sensitivity": target,
                    "auroc": float(roc_auc_score(y, p)), "auprc": float(average_precision_score(y, p))})
        policy_rows[(name, label)] = row

validation_operating_points = pd.DataFrame(list(policy_rows.values()))[
    ["model", "policy", "target_sensitivity", "threshold", "sensitivity", "specificity", "precision", "f1",
     "false_negative", "false_positive", "flagged_percent", "auroc", "auprc"]
]
display(validation_operating_points)

## Seed-to-seed spread of Model B

In [ ]:
b_names = [name for name, info in candidates.items() if info["kind"] == "B"]
seed_table = pd.DataFrame([
    {
        "candidate": name,
        "seed": candidates[name]["seed"],
        "best_epoch": candidates[name]["best_epoch"],
        "aggregation": candidates[name]["aggregation"],
        "auroc": policy_rows[(name, "primary_95")]["auroc"],
        "specificity_at_sens95": policy_rows[(name, "primary_95")]["specificity"],
        "false_positive_at_sens95": policy_rows[(name, "primary_95")]["false_positive"],
    }
    for name in b_names
])
display(seed_table)
seed_summary = {
    metric: {
        "mean": float(seed_table[metric].mean()),
        "sd": float(seed_table[metric].std(ddof=1)) if len(seed_table) > 1 else None,
        "min": float(seed_table[metric].min()),
        "max": float(seed_table[metric].max()),
    }
    for metric in ("auroc", "specificity_at_sens95")
}
reference_a = {
    "auroc": policy_rows[("A", "primary_95")]["auroc"],
    "specificity_at_sens95": policy_rows[("A", "primary_95")]["specificity"],
}
display(pd.DataFrame(seed_summary).T.assign(model_A=pd.Series(reference_a)))

## Apply the pre-specified selection rules and freeze the policy

In [ ]:
spec95 = {name: policy_rows[(name, "primary_95")]["specificity"] for name in candidates}
auroc = {name: policy_rows[(name, "primary_95")]["auroc"] for name in candidates}
BEST_B = max(b_names, key=lambda name: (spec95[name], auroc[name]))
gain = spec95[BEST_B] - spec95["A"]
PIPELINE_MODEL = "B" if gain >= CLOSE_SPECIFICITY_MARGIN else "A"
best = candidates[BEST_B]

operating_policy = {
    "frozen_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "selection_split": "internal_validation",
    "official_mura_valid_read": False,
    "paths_relative_to": "team DATA folder (DATA_DIR)",
    "threshold_rule": "highest internal-validation threshold with sensitivity >= target; flag if probability >= threshold",
    "primary_sensitivity_target": PRIMARY_SENSITIVITY_TARGET,
    "reference_sensitivity_target": REFERENCE_SENSITIVITY_TARGET,
    "model_b_selection_rule": "highest validation specificity at sensitivity >= 95% among Model B seed runs (ties: AUROC)",
    "pipeline_model_rule": f"Model B if its specificity at sensitivity >= 95% exceeds Model A's by >= {CLOSE_SPECIFICITY_MARGIN}; otherwise Model A",
    "pipeline_model": PIPELINE_MODEL,
    "specificity_gain_b_minus_a": float(gain),
    "model_b_candidates": seed_table.to_dict(orient="records"),
    "model_b_seed_summary": seed_summary,
    "models": {
        "A": {
            "description": "Frozen ImageNet ResNet-18 embeddings (mean per study) + Logistic Regression",
            "model_version": "model_a@" + sha256_of(MODEL_A_FILE)[:12],
            "artifact": relative_to_data(MODEL_A_FILE),
            "artifact_sha256": sha256_of(MODEL_A_FILE),
            "validation_predictions": relative_to_data(candidates["A"]["predictions_path"]),
            "validation_predictions_sha256": sha256_of(candidates["A"]["predictions_path"]),
            "primary_threshold": policy_rows[("A", "primary_95")]["threshold"],
            "reference_threshold_90": policy_rows[("A", "reference_90")]["threshold"],
            "validation_auroc": auroc["A"],
            "validation_specificity_at_sens95": spec95["A"],
        },
        "B": {
            "description": f"ImageNet ResNet-18 fine-tuned at study level ({BEST_B})",
            "model_version": "model_b@" + best["artifact_sha256"][:12],
            "run_id": best["run_id"],
            "seed": best["seed"],
            "aggregation": best["aggregation"],
            "artifact": relative_to_data(best["artifact"]),
            "artifact_sha256": best["artifact_sha256"],
            "validation_predictions": relative_to_data(best["predictions_path"]),
            "validation_predictions_sha256": sha256_of(best["predictions_path"]),
            "primary_threshold": policy_rows[(BEST_B, "primary_95")]["threshold"],
            "reference_threshold_90": policy_rows[(BEST_B, "reference_90")]["threshold"],
            "validation_auroc": auroc[BEST_B],
            "validation_specificity_at_sens95": spec95[BEST_B],
        },
    },
}
POLICY_PATH = out_path(RUN_DIR, "operating_policy.json")
POLICY_PATH.write_text(json.dumps(operating_policy, indent=2), encoding="utf-8")
validation_operating_points.to_csv(out_path(RUN_DIR, "validation_operating_points.csv"), index=False)
seed_table.to_csv(out_path(RUN_DIR, "model_b_seed_runs.csv"), index=False)
print(f"Model B for the locked test: {BEST_B} | specificity at sens>=95%: B {spec95[BEST_B]:.3f} vs A {spec95['A']:.3f} (gain {gain:+.3f})")
print("Pipeline model:", PIPELINE_MODEL)
print("Policy:", POLICY_PATH, "| sha256:", sha256_of(POLICY_PATH)[:12])
print()
print("POLICY_RUN_ID =", repr(RUN_DIR.name), " <- use this in Notebooks 06b and 07")